# (a)Data import

In [ ]:
import sqlite3
import numpy as np
import pandas as pd

# Connect to your SQLite database
db_path = "customer_churn.db"
conn = sqlite3.connect(db_path)

# Explicitly load your three original tables into clear pandas DataFrames
df_customer = pd.read_sql("SELECT * FROM db_customer", conn)
df_subscription = pd.read_sql("SELECT * FROM db_subscription", conn)
df_support = pd.read_sql("SELECT * FROM db_support", conn)

conn.close()

print("Data loaded successfully!")


# (b)Data Cleaning

## data cleaning on df_customer dataframe
## 1-rename column -name to customer_name 
## 2-drop columns interests and pincode 
## 3-change data type-column dob 
## 4-data strandardization-column gender 
## 5-fix missing values-column country 

In [ ]:
# 1-rename column
df_customer.rename(columns={"name" : "customer_name"},inplace=True)
df_customer.head(5)

In [ ]:
# 2-drop columns interest and pincode
df_customer.drop(columns=["interests","pincode"],inplace=True)

In [ ]:
#3-change data type-column dob
df_customer['dob'] = pd.to_datetime(df_customer['dob'])
df_customer.info()


In [ ]:
# 4-data strandardization-column gender
df_customer.gender.unique()
df_customer.gender=df_customer.gender.replace({"Male":"Men","Women":"Female"})
df_customer.gender


In [ ]:
# 5-fix missing values-column country
state_country_mapping = df_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()
df_customer['country']=df_customer['country'].fillna(df_customer['state'].map(state_country_mapping))

In [ ]:
df_customer.info()
print(f"df_customer is now cleaned and can be used for further analysis")

### Save Cleaned Customer Data to Database 
### Push the fully cleaned `df_customer` DataFrame into SQLite as a permanent table named `customer_clean` for future SQL queries and analysis.

In [ ]:
import sqlite3
conn = sqlite3.connect(db_path)
df_customer.to_sql("customer_clean", conn, if_exists="replace", index=False)
conn.close()
print("Saved successfully")

In [ ]:
import sqlite3
import pandas as pd

# 1. Connect to your database
conn = sqlite3.connect(db_path)

# 2. Select everything from your clean table
# This query tells SQL to fetch all the data from the table you just created
df_view = pd.read_sql("SELECT * FROM customer_clean", conn)

# 3. Close the connection
conn.close()

# 4. Display the result
df_view

In [ ]:
df_subscription.info()

## data cleaning in df_subscription dataframe
## a-change data type of subscription_start_date,renewal_date and cancellation_date from str to datetime

In [ ]:
df_subscription["subscription_start_date"]=pd.to_datetime(df_subscription["subscription_start_date"])
df_subscription["renewal_date"]=pd.to_datetime(df_subscription["renewal_date"])
df_subscription["cancellation_date"]=pd.to_datetime(df_subscription["cancellation_date"])
df_subscription.info()

### Save Cleaned Subscription Data to Database
### Push the fully cleaned df_customer DataFrame into SQLite as a permanent table named subscription_clean for future SQL queries and analysis.

In [ ]:
import sqlite3
conn = sqlite3.connect(db_path)
df_subscription.to_sql(
    "subscription_clean", conn, if_exists="replace", index=False
)
conn.close()
print("subscription_clean saved successfully!")

In [ ]:
df_support.info()

## Data cleaning in df_support dataframe
## 1-change datatype of complaint_date from str to datetime
## 2-drop columns col_1,comment

In [ ]:
# 1-change datatype of complaint_date from str to datetime¶
df_support["complaint_date"]=pd.to_datetime(df_support["complaint_date"])
df_support.info()


In [ ]:
#2-drop columns col_1,comment¶
df_support.drop(columns=["col_1","comment"],inplace=True)
df_support.info()

### Save Cleaned Support Data to Database
### Push the fully cleaned df_support DataFrame into SQLite as a permanent table named support_clean for future SQL queries and analysis.

In [ ]:
import sqlite3
import pandas as pd
conn=sqlite3.connect(db_path)
df_support.to_sql("support_clean",conn,if_exists="replace",index=False)
conn.close()
print("Saved successfully")


In [ ]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect(db_path)

# Fetch all table names
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
conn.close()

# Display the tables
print("Tables currently in the database:")
print(tables)

## (C) Feature Engineering 

### 1-create a churn flag for subscription_clean table
### 2-as customerid is not unique in support_clean table perform data modeling
### 3-make one master table of all three cleaned table for later analysis

In [ ]:
# 1-create a churn flag for subscription_clean table
import pandas as pd
import sqlite3 
conn = sqlite3.connect(db_path)
query1 = """
    select *, 
        case 
            when cancellation_date is not null then 1
            else 0 
        end as churn_flag
    from subscription_clean
"""
df_subscription_clean_churnflag = pd.read_sql(query1, conn)
df_subscription_clean_churnflag.head()

In [ ]:
# (2)as you can see customerid for support_clean is not unique so have to perform data modeling on this table
import sqlite3
import pandas as pd
conn = sqlite3.connect(db_path)
query1="""select * from support_clean"""
df_not_unique=pd.read_sql(query1,conn)
conn.close()
df_not_unique

In [ ]:
# removing duplicates from customerid
import pandas as pd
import sqlite3

conn = sqlite3.connect(db_path)
query="""
WITH my_cte AS (
    SELECT 
        customerid,
        complaint_date,
        escalations,
        csat_score,
        COUNT(customerid) OVER (PARTITION BY customerid) AS complaint_count,
        ROW_NUMBER() OVER (
            PARTITION BY customerid 
            ORDER BY complaint_date DESC, escalations DESC
        ) AS rn
    FROM support_clean
)
SELECT 
    customerid,
    complaint_date,
    escalations,
    csat_score,
    complaint_count
FROM my_cte 
WHERE rn = 1
"""

df_support_unique = pd.read_sql(query, conn)
conn.close()

df_support_unique

In [ ]:
# (3) as subscription_clean is the one having churn_score,so performing left join on it with rest two table for one master_table
import sqlite3
import pandas as pd
conn = sqlite3.connect(db_path)
df_subscription_clean_churnflag.to_sql("subscription_churn_flag_table",conn,if_exists="replace",index=False)
df_support_unique.to_sql("unique_support_table",conn,if_exists="replace",index=False)
conn.close()


In [ ]:
##one master table of all three tables
import pandas as pd
import sqlite3

conn = sqlite3.connect(db_path)

query = """
    SELECT 
        s.*,
        c.customer_name, c.country, c.state, c.gender, c.dob,
        sup.complaint_date, sup.escalations, sup.csat_score, sup.complaint_count
    FROM subscription_churn_flag_table AS s
    LEFT JOIN customer_clean AS c 
        ON s.customerid = c.customerid
    LEFT JOIN unique_support_table AS sup 
        ON s.customerid = sup.customerid
"""

df_master = pd.read_sql(query, conn)
conn.close()



# (D) Data Analysis

In [ ]:
#final exported csv master table
df_master.to_csv("final_master_table.csv",index=False)

In [ ]:
# making table of df_master df for data analysis
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
df_master.to_sql("master_file_table",conn,if_exists="replace",index=False)
conn.close()


### 1.Churn rate

In [ ]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query="""
select avg(churn_flag)*100 as churn_rate from master_file_table """
churn_rate=pd.read_sql(query,conn)
conn.close()
churn_rate


### 2.Retention rate

In [ ]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query=""" select (100-(avg(churn_flag)*100)) as retention_rate from master_file_table"""
retention_rate=pd.read_sql(query,conn)
conn.close()
retention_rate

### 3.Churn by plan_type


In [ ]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query=""" select plan_type,avg(churn_flag)*100 as churn_rate from master_file_table
group by plan_type
order by churn_rate desc"""
churn_by_plan_type=pd.read_sql(query,conn)
conn.close()
churn_by_plan_type


### 4(a)Churn by state + sum(revenue) and count of users

In [ ]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query="""
select state,avg(churn_flag)*100 as churn_percent,count(customerid) as total_users,sum(monthly_charges) as total_revenue from master_file_table 
group by state
"""
df_by_state=pd.read_sql(query,conn)
conn.close()
df123

### 4(b)Churn by subscription_type + sum(revenue) and count of users

In [ ]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query="""
select subscription_type,avg(churn_flag)*100 as churn_percent,count(customerid) as total_users,sum(monthly_charges) as total_revenue from master_file_table 
group by subscription_type
"""
df_by_subscription=pd.read_sql(query,conn)
conn.close()
df_by_subscription

### 5.ARPU(Average revenue per user)

In [ ]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query="""select avg(monthly_charges) as ARPU from master_file_table"""
Arpu=pd.read_sql(query,conn)
conn.close()
Arpu

### 6.Average tenure days

In [ ]:
import pandas as pd
import sqlite3
conn=sqlite3.connect(db_path)
query = """
    SELECT 
        customerid,
        subscription_start_date,
        cancellation_date,
        CASE 
            WHEN cancellation_date IS NULL OR cancellation_date = '' THEN 
                -- Active user: calculate days from start date to current date
               Round(julianday('now') - julianday(subscription_start_date),0)
            ELSE 
                -- Churned user: calculate days from start date to cancellation date
                Round(julianday(cancellation_date) - julianday(subscription_start_date),0)
        END AS tenure_days
    FROM master_file_table
"""
df_tenure = pd.read_sql(query, conn)
conn.close()
print(f"Average tenure days : {df_tenure['tenure_days'].mean()}")

### 7.Revenue at loss: revenue lost from churned users

In [ ]:
import sqlite3
conn=sqlite3.connect(db_path)
query = """
    SELECT 
        SUM(monthly_charges) AS total_revenue,
        SUM(CASE WHEN churn_flag = 1 THEN monthly_charges ELSE 0 END) AS churned_revenue,
        SUM(CASE WHEN churn_flag = 0 THEN monthly_charges ELSE 0 END) AS retained_revenue
    FROM master_file_table
"""
revenue_breakdown = pd.read_sql(query, conn)
conn.close()
revenue_breakdown

### 8.Escalation rate

In [ ]:
import sqlite3
conn=sqlite3.connect(r"C:\Users\Aryan\Desktop\Data Analytics Python Project by Rishabh Mishra\customer_churn.db")
query = """
    SELECT 
        AVG(CASE WHEN escalations = 'Y' THEN 1.0 ELSE 0.0 END) * 100 AS escalation_percentage
    FROM master_file_table
"""
escalation_rate = pd.read_sql(query, conn)
conn.close()
escalation_rate


### 9.Avg complaint per user

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)
query = """
    SELECT 
        SUM(complaint_count) * 1.0 / COUNT(customerid) AS avg_complaints
    FROM master_file_table
"""
df = pd.read_sql(query, conn)
conn.close()
df

### 10.Correlation between escalation and churn

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)
query = """
    SELECT 
        CASE WHEN escalations = 'Y' THEN 1 ELSE 0 END AS escalations_num,
        churn_flag
    FROM master_file_table
"""
df = pd.read_sql(query, conn)
conn.close()

correlation = df['escalations_num'].corr(df['churn_flag'])
print("Correlation between escalation vs churn is = ", round(correlation, 2))

### 11.Creating a column using churn_score

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)
query = """
select *,case 
when churn_score<50 then "low"
when churn_score >=50 and churn_score<70 then "mid"
when churn_score>=70 then "high"
end as churn_risk
from master_file_table
"""
churn_risk_df=pd.read_sql(query,conn)
conn.close()
churn_risk_df[["churn_score","churn_risk"]].head()